In [38]:
import numpy as np
from mes_packages import *

# Création d'un maillage

In [39]:
mesh = create_mesh_circle_in_square(0.1, 0.3,0.1)
ordre = 2
Nloc = (ordre + 1) * (ordre + 2) // 2
N_glob_DG = nombre_dof_DG(mesh, ordre)


=== Vérification de l'orientation des triangles ===
Nombre total de triangles: 29

Nombre de triangles corrigés: 0/29
Tous les triangles étaient déjà correctement orientés True


# Construction des coordonnées des degrés de liberté en DG

Pour la méthode de Galerkin Discontinue (DG), chaque élément possède ses propres degrés de liberté. Cette fonction calcule les coordonnées $(x, y)$ de chaque DDL dans le maillage.

### Principe

1. **Triangle de référence** : On définit des points d'interpolation sur le triangle de référence
2. **Transformation affine** : On transforme ces points vers chaque triangle réel du maillage

### Transformation géométrique

Pour un triangle réel de sommets $A_1$, $A_2$, $A_3$, la transformation du triangle de référence vers le triangle réel est :

$$
\begin{pmatrix} x \\ y \end{pmatrix} = A_1 + \xi \cdot (A_2 - A_1) + \eta \cdot (A_3 - A_1)
$$

où $(\xi, \eta)$ sont les coordonnées sur le triangle de référence avec $0 \leq \xi, \eta$ et $\xi + \eta \leq 1$.

### Points d'interpolation

Pour un ordre $p$, on considère $N_{loc} = \frac{(p+1)(p+2)}{2}$ points d'interpolation. 

Pour chaque paire d'indices $(m, n)$ avec $0 \leq m, n$ et $m + n \leq p$, le point d'interpolation sur le triangle de référence est :

$$
(\xi, \eta) = \left(\frac{m}{p}, \frac{n}{p}\right)
$$

### Indice global

Pour un élément $i_{elt}$ et un indice local $i_{loc}$, l'indice global est :

$$
i_{glob} = N_{loc} \cdot i_{elt} + i_{loc}
$$

Cette numérotation garantit que les DDL de chaque élément sont consécutifs en mémoire et correspond à ce qui a été fait précédemment


In [40]:
dof_coords_DG = build_dof_coordinates_DG(mesh, ordre)
print_premier_DDL_DG(dof_coords_DG)

Nombre de DDL : 174

Coordonnées des premiers DDL :
DDL 0 : x = 0.017365, y = -0.098481
DDL 1 : x = -0.016318, y = -0.092542
DDL 2 : x = -0.016318, y = -0.124240
DDL 3 : x = -0.050000, y = -0.086603
DDL 4 : x = -0.050000, y = -0.118301
DDL 5 : x = -0.050000, y = -0.150000
DDL 6 : x = -0.050000, y = 0.086603
DDL 7 : x = -0.016318, y = 0.092542
DDL 8 : x = -0.050000, y = 0.118301
DDL 9 : x = 0.017365, y = 0.098481


# Construction des deux matrice mixtes globales en DG qui représentent les formes sesquilinaires
$$
k_x(u,v)\;=\;\int_{\Omega}\frac{\partial u}{\partial x}(x,y)\;\overline{v(x,y)}dxdy
$$
et
$$
k_y(u,v)\;=\;\int_{\Omega}\frac{\partial u}{\partial y}(x)\;\overline{v(x,y)}dxdy
$$

In [41]:
# Déclaration des matrices sparses pour Kx et Ky
Kx_globale = COOMatrix(N_glob_DG, N_glob_DG, 20*Nloc*N_glob_DG)
Ky_globale = COOMatrix(N_glob_DG, N_glob_DG, 20*Nloc*N_glob_DG)

# Assemblage de la matrice de masse de frontière 

## Objectif

Cette fonction assemble la **matrice de masse uniquement sur les faces de bord** (frontière du domaine), c'est-à-dire :

$$
M_{boundary} = \sum_{F \in \mathcal{F}_{bord}} \int_F \varphi_i \varphi_j \, d\sigma
$$

où $\mathcal{F}_{bord}$ est l'ensemble des faces situées sur le bord du domaine.

## Différence importante

⚠️ **Attention** : Cette fonction traite les **faces de bord** (`V < 0`), pas les faces intérieures (`V >= 0`) !

- Si `V >= 0` : face intérieure (entre deux triangles) → **ignorée** (bloc vide dans le if)
- Si `V < 0` : face de bord (sur la frontière du domaine) → **assemblée**

## Processus

1. **Calcul du nombre de faces de bord** : `np.sum(neighbors < 0)` compte les entrées négatives dans `neighbors`

2. **Estimation de la mémoire** : `nnz = n_faces_bord * (ordre + 1)²` 
   - Chaque face de bord a `ordre + 1` DDL
   - Contribution de `(ordre + 1)²` éléments matriciels par face

3. **Assemblage** :
   - Parcourt tous les triangles et leurs faces
   - Si `V < 0` (face de bord) : construit `MTT` et l'ajoute à `M_boundary`
   - Si `V >= 0` (face intérieure) : ne fait rien


In [42]:
M_Gamma = build_matrice_masse_frontière_DG(mesh,ordre)